# Project 7: User/Data Interaction (HCI, M2M, UI, UX)
## Employee Performance Analytics Dashboard

**Course:** Modern Database Technologies and Big Data Analytics  
**Institution:** Transport and Telecommunication Institute, Latvia  
**Level:** Master's Program

This notebook demonstrates HCI principles through an interactive dashboard with:
- Direct Manipulation (widgets, interactive charts)
- Form-Based Interaction (structured data entry)
- Natural Language Interface (query-based exploration)
- Visual Feedback (real-time updates)

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Data Generation

In [2]:
def generate_employee_data(n=150, seed=42):
    """Generate synthetic employee performance dataset."""
    np.random.seed(seed)
    departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance', 'Operations', 'R&D', 'Support']
    positions = {'Engineering': ['Jr Dev', 'Sr Dev', 'Tech Lead'], 'Sales': ['Sales Rep', 'Account Mgr'],
                 'Marketing': ['Analyst', 'Manager'], 'HR': ['Specialist', 'Manager'],
                 'Finance': ['Analyst', 'Accountant'], 'Operations': ['Analyst', 'Manager'],
                 'R&D': ['Scientist', 'Researcher'], 'Support': ['Agent', 'Lead']}
    education = ['High School', "Bachelor's", "Master's", 'PhD']
    
    data = []
    for i in range(n):
        dept = np.random.choice(departments, p=[0.25, 0.15, 0.12, 0.08, 0.1, 0.1, 0.1, 0.1])
        base_perf = np.random.normal(70, 15)
        data.append({
            'Employee_ID': f'EMP{1000+i}', 'Name': f'Employee_{i+1}',
            'Department': dept, 'Position': np.random.choice(positions[dept]),
            'Education': np.random.choice(education, p=[0.15, 0.45, 0.30, 0.10]),
            'Years_Experience': np.random.randint(1, 25), 'Age': np.random.randint(22, 60),
            'Performance_Score': round(np.clip(base_perf + np.random.normal(0, 5), 0, 100), 1),
            'Satisfaction_Score': round(np.clip(base_perf * 0.8 + np.random.normal(20, 10), 0, 100), 1),
            'Projects_Completed': np.random.poisson(8), 'Training_Hours': np.random.randint(10, 200),
            'Salary': np.random.randint(35000, 150000),
            'Manager_Rating': round(np.clip(base_perf * 0.9 + np.random.normal(10, 8), 0, 100), 1),
            'Attendance_Rate': round(np.clip(95 + np.random.normal(0, 3), 80, 100), 1)
        })
    return pd.DataFrame(data)

df = generate_employee_data()
print(f"Dataset: {len(df)} employees, {len(df.columns)} columns")
df.head()

Dataset: 150 employees, 14 columns


,Employee_ID,Name,Department,Position,Education,Years_Experience,Age,Performance_Score,Satisfaction_Score,Projects_Completed,Training_Hours,Salary,Manager_Rating,Attendance_Rate
0,EMP1000,Employee_1,Sales,Sales Rep,High School,11,45,54.9,53.8,4,197,139724,54.7,99.4
1,EMP1001,Employee_2,Engineering,Jr Dev,Bachelor's,10,49,94.2,86.8,8,18,58897,96.1,93.1
2,EMP1002,Employee_3,Marketing,Analyst,Bachelor's,17,57,64.5,81.5,6,23,67606,67.2,92.0
3,EMP1003,Employee_4,Sales,Sales Rep,Bachelor's,13,50,49.2,65.9,7,57,70222,61.3,98.7
4,EMP1004,Employee_5,HR,Specialist,High School,14,24,68.4,85.1,5,72,85015,91.5,90.7


## 2. Interaction Logging System
**HCI Principle:** Visibility of System Status

In [ ]:
class InteractionLogger:
    def __init__(self):
        self.log = []
        self.start = datetime.now()
    
    def record(self, action, details):
        self.log.append({'time': datetime.now().isoformat(), 'action': action, 'details': details})
    
    def summary(self):
        if not self.log:
            return "No interactions recorded."
        df_log = pd.DataFrame(self.log)
        return {'total': len(self.log), 'actions': df_log['action'].value_counts().to_dict()}

logger = InteractionLogger()
logger.record('session_start', 'Notebook initialized')
print("Interaction Logger ready!")

Interaction Logger ready!


## 3. KPI Display Function
**HCI Principle:** Visibility of System Status

In [4]:
def display_kpis(data, title="Key Performance Indicators"):
    logger.record('kpi_view', f'{len(data)} records')
    kpis = {'Employees': len(data), 'Avg Performance': f"{data['Performance_Score'].mean():.1f}%",
            'Avg Satisfaction': f"{data['Satisfaction_Score'].mean():.1f}%",
            'Total Projects': data['Projects_Completed'].sum(), 'Avg Salary': f"${data['Salary'].mean():,.0f}"}
    
    html = f"<h3 style='color:#1E3A5F;border-bottom:2px solid #3498DB;padding-bottom:10px;'>{title}</h3>"
    html += "<div style='display:flex;flex-wrap:wrap;gap:15px;margin-top:10px;'>"
    for label, value in kpis.items():
        html += f"""<div style='background:linear-gradient(135deg,#667eea,#764ba2);border-radius:10px;
                    padding:15px;min-width:120px;text-align:center;color:white;'>
                    <div style='font-size:12px;opacity:0.9;'>{label}</div>
                    <div style='font-size:20px;font-weight:bold;margin-top:5px;'>{value}</div></div>"""
    html += "</div>"
    display(HTML(html))

display_kpis(df)

## 4. Interactive Filtering System
**HCI Principles:** User Control, Direct Manipulation, Recognition vs. Recall

In [5]:
# Filter Widgets
dept_filter = widgets.SelectMultiple(options=['All'] + sorted(df['Department'].unique().tolist()),
                                     value=['All'], description='Dept:', layout=widgets.Layout(width='250px', height='100px'))
perf_filter = widgets.FloatRangeSlider(value=[0, 100], min=0, max=100, step=5,
                                       description='Perf:', layout=widgets.Layout(width='350px'), continuous_update=False)
exp_filter = widgets.IntRangeSlider(value=[1, 25], min=1, max=25, step=1,
                                    description='Exp:', layout=widgets.Layout(width='350px'), continuous_update=False)
filter_output = widgets.Output()

def apply_filters(change=None):
    with filter_output:
        clear_output(wait=True)
        filtered = df.copy()
        if 'All' not in dept_filter.value and len(dept_filter.value) > 0:
            filtered = filtered[filtered['Department'].isin(dept_filter.value)]
        filtered = filtered[(filtered['Performance_Score'] >= perf_filter.value[0]) & 
                           (filtered['Performance_Score'] <= perf_filter.value[1])]
        filtered = filtered[(filtered['Years_Experience'] >= exp_filter.value[0]) & 
                           (filtered['Years_Experience'] <= exp_filter.value[1])]
        logger.record('filter', f'{len(filtered)} records')
        color = '#27AE60' if len(filtered) > 0 else '#E74C3C'
        display(HTML(f"<div style='background:#E8F4FD;border-left:4px solid {color};padding:10px;margin:10px 0;'>"
                     f"<strong>Results:</strong> {len(filtered)} of {len(df)} employees ({len(filtered)/len(df)*100:.1f}%)</div>"))
        display_kpis(filtered, "Filtered KPIs")
        return filtered

dept_filter.observe(apply_filters, names='value')
perf_filter.observe(apply_filters, names='value')
exp_filter.observe(apply_filters, names='value')

reset_btn = widgets.Button(description='Reset', button_style='warning')
def reset(b):
    dept_filter.value = ['All']
    perf_filter.value = [0, 100]
    exp_filter.value = [1, 25]
    logger.record('filter_reset', 'all')
    apply_filters()
reset_btn.on_click(reset)

print("INTERACTIVE FILTERS")
print("="*40)
display(widgets.VBox([widgets.HBox([dept_filter, reset_btn]), perf_filter, exp_filter, filter_output]))
apply_filters()

INTERACTIVE FILTERS


,Employee_ID,Name,Department,Position,Education,Years_Experience,Age,Performance_Score,Satisfaction_Score,Projects_Completed,Training_Hours,Salary,Manager_Rating,Attendance_Rate
0,EMP1000,Employee_1,Sales,Sales Rep,High School,11,45,54.9,53.8,4,197,139724,54.7,99.4
1,EMP1001,Employee_2,Engineering,Jr Dev,Bachelor's,10,49,94.2,86.8,8,18,58897,96.1,93.1
2,EMP1002,Employee_3,Marketing,Analyst,Bachelor's,17,57,64.5,81.5,6,23,67606,67.2,92.0
3,EMP1003,Employee_4,Sales,Sales Rep,Bachelor's,13,50,49.2,65.9,7,57,70222,61.3,98.7
4,EMP1004,Employee_5,HR,Specialist,High School,14,24,68.4,85.1,5,72,85015,91.5,90.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,EMP1145,Employee_146,Engineering,Jr Dev,Bachelor's,17,33,76.5,73.5,9,50,146541,74.9,100.0
146,EMP1146,Employee_147,Engineering,Sr Dev,PhD,15,48,100.0,100.0,9,57,148530,100.0,96.8
147,EMP1147,Employee_148,Sales,Sales Rep,Bachelor's,17,25,89.8,98.8,11,162,129574,87.1,90.2
148,EMP1148,Employee_149,Operations,Analyst,Bachelor's,9,41,69.8,82.1,5,52,40039,64.2,94.3


## 5. Data Visualizations
**HCI Principles:** Direct Manipulation, Aesthetic Design, Visual Feedback

In [6]:
# Department Analysis
logger.record('visualization', 'department_analysis')
dept_stats = df.groupby('Department').agg({'Performance_Score': 'mean', 'Employee_ID': 'count'}).reset_index()
dept_stats.columns = ['Department', 'Avg Performance', 'Count']

fig = make_subplots(rows=1, cols=2, subplot_titles=('Employee Count', 'Avg Performance'))
fig.add_trace(go.Bar(x=dept_stats['Department'], y=dept_stats['Count'], marker_color='#3498DB',
                     text=dept_stats['Count'], textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=dept_stats['Department'], y=dept_stats['Avg Performance'],
                     marker_color=dept_stats['Avg Performance'], marker_colorscale='RdYlGn',
                     text=dept_stats['Avg Performance'].round(1), textposition='outside'), row=1, col=2)
fig.update_layout(title='Department Analysis', showlegend=False, height=400)
fig.update_xaxes(tickangle=-45)
fig.show()

In [7]:
# Experience vs Performance Scatter
logger.record('visualization', 'scatter_plot')
fig = px.scatter(df, x='Years_Experience', y='Performance_Score', color='Department', size='Salary',
                 hover_data=['Name', 'Position'], title='Experience vs Performance',
                 labels={'Years_Experience': 'Experience (Years)', 'Performance_Score': 'Performance (%)'})
fig.update_layout(height=500)
fig.show()

In [8]:
# Correlation Heatmap
logger.record('visualization', 'correlation_matrix')
numeric_cols = ['Performance_Score', 'Satisfaction_Score', 'Years_Experience', 'Salary', 'Manager_Rating', 'Attendance_Rate']
corr = df[numeric_cols].corr()
fig = px.imshow(corr, labels=dict(color="Correlation"), color_continuous_scale='RdBu_r',
                title='Performance Metrics Correlation Matrix')
fig.update_layout(height=500)
fig.show()

In [9]:
# Performance Distribution by Education
logger.record('visualization', 'box_plot')
fig = px.box(df, x='Education', y='Performance_Score', color='Education',
             title='Performance Distribution by Education',
             category_orders={'Education': ['High School', "Bachelor's", "Master's", 'PhD']})
fig.update_layout(height=400, showlegend=False)
fig.show()

## 6. Data Explorer
**HCI Principles:** Recognition vs. Recall, Direct Manipulation

In [10]:
search = widgets.Text(placeholder='Search name/ID...', description='Search:', layout=widgets.Layout(width='250px'))
sort_by = widgets.Dropdown(options=['Performance_Score', 'Salary', 'Years_Experience', 'Name'], 
                           value='Performance_Score', description='Sort:')
sort_dir = widgets.ToggleButtons(options=['Desc', 'Asc'], value='Desc')
rows = widgets.IntSlider(value=10, min=5, max=30, description='Rows:')
table_out = widgets.Output()

def update_table(change=None):
    with table_out:
        clear_output(wait=True)
        result = df.copy()
        if search.value:
            result = result[result['Name'].str.contains(search.value, case=False) | 
                           result['Employee_ID'].str.contains(search.value, case=False)]
            logger.record('search', search.value)
        result = result.sort_values(sort_by.value, ascending=(sort_dir.value == 'Asc'))
        cols = ['Employee_ID', 'Name', 'Department', 'Position', 'Performance_Score', 'Salary']
        display(HTML(f"<p><strong>Showing {min(rows.value, len(result))} of {len(result)} records</strong></p>"))
        display(result[cols].head(rows.value).style.background_gradient(subset=['Performance_Score'], cmap='RdYlGn'))

search.observe(update_table, names='value')
sort_by.observe(update_table, names='value')
sort_dir.observe(update_table, names='value')
rows.observe(update_table, names='value')

print("DATA EXPLORER")
print("="*40)
display(widgets.VBox([widgets.HBox([search, sort_by, sort_dir]), rows, table_out]))
update_table()

DATA EXPLORER


## 7. Natural Language Query Interface
**HCI Principle:** Match Between System and Real World

In [11]:
query_input = widgets.Text(placeholder='e.g., "Show top performers"', description='Ask:', layout=widgets.Layout(width='400px'))
quick = widgets.Dropdown(options=['-- Quick queries --', 'Show top 10 performers', 'Average by department',
                                   'High performers (>80)', 'Low satisfaction (<60)'], description='Quick:')
query_out = widgets.Output()

def process(query):
    q = query.lower()
    if 'top' in q and 'performer' in q:
        return "Top 10 Performers", df.nlargest(10, 'Performance_Score')[['Employee_ID', 'Name', 'Department', 'Performance_Score']]
    elif 'average' in q and 'department' in q:
        r = df.groupby('Department')['Performance_Score'].mean().reset_index().sort_values('Performance_Score', ascending=False)
        r.columns = ['Department', 'Avg Performance']
        return "Average by Department", r
    elif '>80' in q or 'high' in q:
        r = df[df['Performance_Score'] >= 80][['Employee_ID', 'Name', 'Department', 'Performance_Score']]
        return f"High Performers ({len(r)} found)", r.sort_values('Performance_Score', ascending=False)
    elif '<60' in q or 'low' in q:
        r = df[df['Satisfaction_Score'] < 60][['Employee_ID', 'Name', 'Department', 'Satisfaction_Score']]
        return f"Low Satisfaction ({len(r)} found)", r
    return None, None

def run_query(change=None):
    with query_out:
        clear_output(wait=True)
        q = query_input.value if query_input.value else (quick.value if quick.value != '-- Quick queries --' else '')
        if not q: return
        logger.record('nl_query', q)
        title, result = process(q)
        if result is not None:
            display(HTML(f"<div style='background:#D4EDDA;border-left:4px solid #27AE60;padding:10px;margin:10px 0;'>"
                        f"<strong>Query:</strong> {q}</div><h4>{title}</h4>"))
            display(result)
        else:
            display(HTML(f"<div style='background:#F8D7DA;border-left:4px solid #E74C3C;padding:10px;'>"
                        f"<strong>Not understood:</strong> {q}<br><em>Try a quick query.</em></div>"))

btn = widgets.Button(description='Execute', button_style='primary')
btn.on_click(run_query)
quick.observe(run_query, names='value')

print("NATURAL LANGUAGE QUERIES")
print("="*40)
display(widgets.VBox([widgets.HBox([query_input, btn]), quick, query_out]))

NATURAL LANGUAGE QUERIES


## 8. Form-Based Data Entry
**HCI Principles:** Error Prevention, Feedback, Constraints

In [12]:
name_input = widgets.Text(placeholder='Full name', description='Name*:')
dept_input = widgets.Dropdown(options=[''] + sorted(df['Department'].unique().tolist()), description='Dept*:')
perf_input = widgets.FloatSlider(value=70, min=0, max=100, description='Perf*:')
salary_input = widgets.IntText(value=50000, description='Salary:')
form_out = widgets.Output()

def submit(b):
    with form_out:
        clear_output(wait=True)
        errors = []
        if not name_input.value: errors.append("Name required")
        if not dept_input.value: errors.append("Department required")
        if errors:
            for e in errors:
                display(HTML(f"<div style='color:#E74C3C;'>❌ {e}</div>"))
            logger.record('form_error', str(errors))
        else:
            display(HTML(f"<div style='background:#D4EDDA;padding:15px;border-radius:8px;'>"
                        f"<strong>✅ Submitted!</strong><br>Name: {name_input.value}<br>"
                        f"Dept: {dept_input.value}<br>Performance: {perf_input.value}%<br>Salary: ${salary_input.value:,}</div>"))
            logger.record('form_submit', name_input.value)

submit_btn = widgets.Button(description='Submit', button_style='success')
submit_btn.on_click(submit)

print("EMPLOYEE ENTRY FORM")
print("="*40)
display(widgets.VBox([name_input, dept_input, perf_input, salary_input, submit_btn, form_out]))

EMPLOYEE ENTRY FORM


## 9. Interaction Log Analysis
**HCI Principle:** Visibility for Usability Analysis

In [13]:
print("INTERACTION LOG SUMMARY")
print("="*40)
summary = logger.summary()
print(f"Total Interactions: {summary['total']}")
print(f"\nAction Distribution:")
for action, count in summary['actions'].items():
    print(f"  {action}: {count}")

if len(logger.log) > 0:
    log_df = pd.DataFrame(logger.log)
    fig = px.pie(log_df, names='action', title='Interaction Distribution')
    fig.update_layout(height=350)
    fig.show()

INTERACTION LOG SUMMARY
Total Interactions: 8

Action Distribution:
  visualization: 4
  kpi_view: 2
  session_start: 1
  filter: 1


## 10. Summary

This notebook demonstrated key HCI principles:

| Principle | Implementation |
|-----------|----------------|
| Visibility of System Status | KPI displays, filter status, loading feedback |
| Match System & Real World | Natural language queries, familiar terminology |
| User Control & Freedom | Reset buttons, undo capability |
| Consistency | Unified styling, predictable behavior |
| Error Prevention | Form validation, constrained inputs |
| Recognition vs. Recall | Dropdowns, quick queries, tooltips |
| Flexibility & Efficiency | Multiple interaction modes |
| Aesthetic Design | Clean layout, visual hierarchy |
| Help & Documentation | Inline help, feedback messages |

**For full Streamlit application:** Run `streamlit run app.py`